# 03 · Contratos de dados com SHACL

**Objetivo:** separar duas coisas que costumam ser confundidas:

* a **ontologia** diz o que as coisas **significam** (uma parada é um evento que acontece numa máquina);
* o **SHACL** diz o que um registro **precisa conter** para ser aceito (toda parada não planejada precisa de um motivo).

Sem contrato, o grafo vira um depósito de dados incompletos, e a primeira consulta de Pareto mostra 40 % de paradas "sem motivo".

In [1]:
# Configuração: deixa a raiz do repositório no caminho de import
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from lab.graph import load_ontology, load_plant, query, MFG, PRD, MNT, QA, NRG, INV, ID
from rdflib import Graph
from lab.graph import ROOT
from lab.validate import validate

## 1. As regras

Um trecho do arquivo de shapes. Cada `sh:property` é uma regra sobre um caminho, e cada `sh:sparql` é uma regra que precisa olhar mais de um campo.

In [2]:
print(open(ROOT / "shapes" / "shapes.ttl", encoding="utf-8").read()[700:2600])

to the graph.
# ---------------------------------------------------------------------------

shp:EquipmentShape a sh:NodeShape ;
    sh:targetClass mfg:Equipment ;
    sh:property [ sh:path mfg:assetId ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:string ;
                  sh:message "Equipment needs exactly one assetId (the identity shared with UNS, MES and CMMS)." ] ;
    sh:property [ sh:path mfg:partOf ; sh:minCount 1 ; sh:class mfg:Line ;
                  sh:message "Equipment must belong to a Line." ] .

shp:DataPointShape a sh:NodeShape ;
    sh:targetClass mfg:DataPoint ;
    sh:property [ sh:path mfg:unsTopic ; sh:minCount 1 ;
                  sh:pattern "^[a-z0-9-]+(/[a-z0-9.-]+)+$" ;
                  sh:message "Every data point needs a UNS topic: lowercase, levels separated by '/'." ] .

shp:WorkOrderShape a sh:NodeShape ;
    sh:targetClass prd:WorkOrder ;
    sh:property [ sh:path prd:forProduct ; sh:minCount 1 ; sh:class prd:Product ;
                  sh:message

## 2. Validando a planta de exemplo

In [3]:
planta = Graph().parse(ROOT / "data" / "plant.ttl")
ok, violacoes = validate(planta)
print("a planta respeita todos os contratos:", ok, "| violações:", len(violacoes))

a planta respeita todos os contratos: True | violações: 0


## 3. Validando o que uma integração descuidada mandaria

`data/broken.ttl` traz cinco registros plausíveis e errados: uma máquina nova sem identidade, um tópico copiado do símbolo do CLP, uma parada com horários trocados e sem motivo, uma ordem sem produto e uma inspeção com resultado em texto livre.

In [4]:
print(open(ROOT / "data" / "broken.ttl", encoding="utf-8").read())

# Records that a careless integration could send. Every one breaks a shape.
@prefix mfg:  <https://w3id.org/industrial-ontology-lab/core#> .
@prefix prd:  <https://w3id.org/industrial-ontology-lab/production#> .
@prefix mnt:  <https://w3id.org/industrial-ontology-lab/maintenance#> .
@prefix qa:   <https://w3id.org/industrial-ontology-lab/quality#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
@prefix id:   <https://w3id.org/industrial-ontology-lab/id/> .

# 1. new machine with no identity and no line
<https://w3id.org/industrial-ontology-lab/id/new-palletizer> a mfg:Equipment ; rdfs:label "Palletizer (new)" .

# 2. tag with a UNS topic copied from a PLC symbol
<https://w3id.org/industrial-ontology-lab/id/new-palletizer/speed> a mfg:DataPoint ;
    mfg:unsTopic "PLC1.DB10.Speed" .

# 3. downtime typed in with start and end swapped and no reason
<https://w3id.org/industrial-ontology-lab/id/downtime/D-999> a mnt:Downtime ;
  

In [5]:
sujo = Graph().parse(ROOT / "data" / "plant.ttl")
sujo.parse(ROOT / "data" / "broken.ttl")
ok, violacoes = validate(sujo)
print("conforma:", ok, "| violações:", len(violacoes))
pd.DataFrame(violacoes)

conforma:

 False | violações: 8


,focus,path,message
0,downtime/D-999,,End time must be after start time.
1,downtime/D-999,,"An unplanned downtime must have a failure mode (causedBy), otherwise the Par..."
2,inspection/Q-9999,result,"Inspection result must be PASS, HOLD or REJECT."
3,new-palletizer,assetId,"Equipment needs exactly one assetId (the identity shared with UNS, MES and C..."
4,new-palletizer,partOf,Equipment must belong to a Line.
5,new-palletizer/speed,unsTopic,"Every data point needs a UNS topic: lowercase, levels separated by '/'."
6,work-order/WO-9999,forProduct,A work order must say which product it makes.
7,work-order/WO-9999,targetQty,Target quantity must be positive.


**Leitura, violação por violação**

* `new-palletizer` sem `assetId` e sem `partOf`: a máquina não tem identidade compartilhada com UNS, MES e CMMS, e não está em nenhuma linha. Ela nunca apareceria num OEE por linha.
* `new-palletizer/speed` com tópico `PLC1.DB10.Speed`: isso é o endereço do CLP, não um tópico do namespace. O contrato exige o caminho ISA-95 em minúsculas.
* `D-999`: fim antes do início **e** parada não planejada sem motivo. São duas regras que olham mais de um campo, por isso são `sh:sparql`.
* `WO-9999`: ordem sem produto e com quantidade zero.
* `Q-9999`: resultado "ok, looks fine" em vez de PASS, HOLD ou REJECT. Texto livre quebra qualquer indicador de qualidade.

## 4. Onde isso entra numa arquitetura real

O contrato roda **na entrada** do grafo: no conector do MES, no fluxo do Node-RED, no job de DataOps. Registro que falha vai para uma fila de correção com a mensagem da regra, em vez de entrar silenciosamente.

In [6]:
def aceitar(registro_ttl: str, grafo: Graph):
    candidato = Graph(); [candidato.add(t) for t in grafo]
    candidato.parse(data=registro_ttl, format="turtle")
    ok, v = validate(candidato)
    if ok:
        grafo.parse(data=registro_ttl, format="turtle"); return "aceito", []
    novos = [x for x in v if x["focus"] in registro_ttl]
    return "rejeitado", [x["message"] for x in novos]

base = Graph().parse(ROOT / "data" / "plant.ttl")
bom = '''@prefix mnt: <https://w3id.org/industrial-ontology-lab/maintenance#> . @prefix mfg: <https://w3id.org/industrial-ontology-lab/core#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
<https://w3id.org/industrial-ontology-lab/id/downtime/D-100> a mnt:Downtime ; mnt:planned false ;
  mnt:causedBy <https://w3id.org/industrial-ontology-lab/id/failure-mode/jam> ;
  mfg:occurredOn <https://w3id.org/industrial-ontology-lab/id/acme/sao-paulo/packaging/line-2/filler-01> ;
  mfg:startTime "2026-11-10T14:00:00+00:00"^^xsd:dateTime ; mfg:endTime "2026-11-10T14:05:00+00:00"^^xsd:dateTime .'''
ruim = bom.replace("D-100", "D-101").replace("  mnt:causedBy <https://w3id.org/industrial-ontology-lab/id/failure-mode/jam> ;\n", "")
print("D-100:", aceitar(bom, base))
print("D-101:", aceitar(ruim, base))

D-100: ('aceito', [])
D-101: ('rejeitado', ['An unplanned downtime must have a failure mode (causedBy), otherwise the Pareto is useless.'])


## Resumo

* **Ontologia = significado**, **SHACL = contrato**. As duas são necessárias.
* Regras simples (`minCount`, `pattern`, `in`, `class`) cobrem a maioria dos casos; regras que comparam campos usam `sh:sparql`.
* Validar na entrada mantém o grafo confiável. Um grafo com dado ruim é pior que nenhum, porque as consultas parecem certas.